In [2]:
from column import TNN_Col
from layer import Layer
from submodule import *
from backend.backend import * 
import argparse
import os
from rich.console import Console
from tnn_mdls.func_mdls import *
from tnn_mdls.tb_func_mdls import *
import time
from veriloggen import *
import copy
from model import Model
import numpy as np

In [2]:
myModel = Model()

In [3]:
myModel.add(Layer(layer_type="TNN", num_col=16, num_neurons=2, num_dend=1, p_dist=18, p_prox=1, num_seg=1, wres_dist=3, wres_prox=3, thres=6))

In [4]:
myModel.add(Layer(layer_type="Kernel", rfsize=2, stride=1, nprev=2, inputsize=4))

In [5]:
myModel.add(Layer(layer_type="TNN", num_col=9, num_neurons=1, num_dend=1, p_dist=8, p_prox=1, num_seg=1, wres_dist=3, wres_prox=3, thres=6))

In [6]:
myModel.compile()

In [8]:
gen_file, rtl_path = gen_verilog(module = myModel.model, filename = 'kernel_model.v')

In [7]:
myModel.summary()

TNN2_Layer_0
     NUM_COL 16
     NUM_NEURONS 2
     NUM_DEND 1
     P_DIST 18
     P_PROX 1
     NUM_SEG 1
     WRES_DIST 3
     WRES_PROX 3
     THRESHOLD 6
     IN_WIDTH 288
     OUT_WIDTH 32
     is_clk 1
Kernel_Layer_1
     RFSIZE 2
     STRIDE 1
     NPREV 2
     INPUTSIZE 4
     IN_WIDTH 32
     p 8
     OUT_WIDTH 72
     is_clk 0
TNN2_Layer_2
     NUM_COL 9
     NUM_NEURONS 1
     NUM_DEND 1
     P_DIST 8
     P_PROX 1
     NUM_SEG 1
     WRES_DIST 3
     WRES_PROX 3
     THRESHOLD 6
     IN_WIDTH 72
     OUT_WIDTH 9
     is_clk 1


In [7]:
num_col = 2
num_neurons = 12
p_dist = 4

for c in range(num_col):
    print((c+1)*num_neurons-1, c*num_neurons)


11 0
23 12


In [9]:
np.set_printoptions(edgeitems=300, linewidth=1000000, 
    formatter=None)

In [9]:
row = 4
col = 5
m = np.zeros((row,col))

In [10]:
count = 0
for i in range(row):
    for j in range(col):
        m[i][j] = count
        count += 1

In [11]:
m

array([[ 0.,  1.,  2.,  3.,  4.],
       [ 5.,  6.,  7.,  8.,  9.],
       [10., 11., 12., 13., 14.],
       [15., 16., 17., 18., 19.]])

In [12]:
n = m.flatten()
n

array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.,
       13., 14., 15., 16., 17., 18., 19.])

In [13]:
for i in range(4):
    for j in range(5):
        print(m[i][j])

0.0
1.0
2.0
3.0
4.0
5.0
6.0
7.0
8.0
9.0
10.0
11.0
12.0
13.0
14.0
15.0
16.0
17.0
18.0
19.0


In [21]:
for i in range(4):
    for j in range(5):
        print(n[i*5])

0.0
0.0
0.0
0.0
0.0
5.0
5.0
5.0
5.0
5.0
10.0
10.0
10.0
10.0
10.0
15.0
15.0
15.0
15.0
15.0


In [12]:
m*12

array([[  0.,  12.,  24.,  36.],
       [ 48.,  60.,  72.,  84.],
       [ 96., 108., 120., 132.],
       [144., 156., 168., 180.]])

In [13]:
arr = np.arange(row*col)

In [14]:
k_dim = 2
n_col = col - (k_dim-1)
n_row = row - (k_dim-1)
stride = 2
num_col = n_col*n_row
nprev = 12

In [15]:
num_col, col, row, n_col

(9, 4, 4, 3)

In [16]:
for c in range(0, num_col, stride):
    ky = int(np.floor(c/n_col))
    kx = int(np.floor(c%n_row))
    #slices = []
    for s in range(k_dim):
        #slices += [((kx + ky*col + col*s)*nprev, ((kx + ky*col + col*s + (k_dim-1)))*nprev+(nprev-1))]
        print("Slice "+str(s+1)+":", (kx + ky*col + col*s)*num_neurons, ((kx + ky*col + col*s + (k_dim-1)))*num_neurons+(num_neurons-1))

Slice 1: 0 23
Slice 2: 48 71
Slice 1: 24 47
Slice 2: 72 95
Slice 1: 60 83
Slice 2: 108 131
Slice 1: 96 119
Slice 2: 144 167
Slice 1: 120 143
Slice 2: 168 191


In [144]:
slices[0]

(7776, 7799)

In [116]:
kernel_count = 0
k_dim = 2
n_col = col - (k_dim-1)
n_row = row - (k_dim-1)
stride = 1
new_arr = []

for ky in range(0, n_row, stride):
    for kx in range(0, n_col, stride):
        kernel = np.zeros((k_dim,k_dim))
        for s in range(k_dim):
            print("Slice "+str(s+1)+":", (kx + ky*col + col*s)*num_neurons, ((kx + ky*col + col*s + (k_dim-1)))*num_neurons+(num_neurons-1))
            
        for x in range(k_dim):
            for y in range(k_dim):
                kernel[x][y] = y + x*col + kx + ky*col
                new_arr += [arr[y + x*col + kx + ky*col]]
                #print(arr[y + x*col + kx + ky*col], end = ", ")
                #print("Slices:", y + x*col + kx + ky*col)
                
        #print(kernel)
        #print("Slices:", y + x*col + kx + ky*col)
        kernel_count += 1
        kx += stride
    ky += stride

Slice 1: 0 23
Slice 2: 120 143
Slice 1: 12 35
Slice 2: 132 155
Slice 1: 24 47
Slice 2: 144 167
Slice 1: 36 59
Slice 2: 156 179
Slice 1: 48 71
Slice 2: 168 191
Slice 1: 60 83
Slice 2: 180 203
Slice 1: 72 95
Slice 2: 192 215
Slice 1: 84 107
Slice 2: 204 227
Slice 1: 96 119
Slice 2: 216 239
Slice 1: 120 143
Slice 2: 240 263
Slice 1: 132 155
Slice 2: 252 275
Slice 1: 144 167
Slice 2: 264 287
Slice 1: 156 179
Slice 2: 276 299
Slice 1: 168 191
Slice 2: 288 311
Slice 1: 180 203
Slice 2: 300 323
Slice 1: 192 215
Slice 2: 312 335
Slice 1: 204 227
Slice 2: 324 347
Slice 1: 216 239
Slice 2: 336 359
Slice 1: 240 263
Slice 2: 360 383
Slice 1: 252 275
Slice 2: 372 395
Slice 1: 264 287
Slice 2: 384 407
Slice 1: 276 299
Slice 2: 396 419
Slice 1: 288 311
Slice 2: 408 431
Slice 1: 300 323
Slice 2: 420 443
Slice 1: 312 335
Slice 2: 432 455
Slice 1: 324 347
Slice 2: 444 467
Slice 1: 336 359
Slice 2: 456 479
Slice 1: 360 383
Slice 2: 480 503
Slice 1: 372 395
Slice 2: 492 515
Slice 1: 384 407
Slice 2: 504 5

In [112]:
kernel_count

64

In [224]:
len(new_arr)

576

In [8]:
L = Layer(layer_type="Kernel", rfsize=2, nprev=12, inputsize=4, stride=1)
m = L.Kernel_Layer(0)

In [9]:
gen_file, rtl_path = gen_verilog(module = m, filename = 'kernel.v')

In [219]:
layer_id = str(0)
num_col = m.Parameter('NUM_COL', int(2))
num_neurons = m.Parameter('NUM_NEURONS', int(2))
num_dend = m.Parameter('NUM_DEND', int(2))
p_dist = m.Parameter('P_DIST', int(9))
p_prox = m.Parameter('P_PROX', int(1))
num_seg = m.Parameter('NUM_SEG', int(2))
wres_dist = m.Parameter('WRES_DIST', int(3))
wres_prox = m.Parameter('WRES_PROX', int(3))
threshold = m.Parameter('THRESHOLD', int(6))

tnn_func = TNN_Functions(layer_id)
comp_col, _ = tnn_func.Comp_column(num_neurons.value, num_dend.value, p_dist.value, p_prox.value, num_seg.value, wres_dist.value, wres_prox.value, threshold.value)

In [220]:
in1 = m.Input('in1', int(9))
temp_port = Cat(in1.slice(0,2), in1.slice(3,5), in1.slice(6,8))

In [221]:
m.Instance(comp_col, 'L'+str(layer_id)+'_comp_col_inst_'+str(c), params=[num_neurons.value, num_dend.value, p_dist.value, p_prox.value, num_seg.value, wres_dist.value, wres_prox.value, threshold.value],
                       ports = [temp_port])

In [205]:
Cat(in1.slice(0,3), in1.slice(3,6))

In [2]:
myModel = Model()
myModel.add(Layer(layer_type="TNN", num_col=2, num_neurons=2, num_dend=1, p_dist=18, p_prox=1, num_seg=4, wres_dist=3, wres_prox=3, thres=6))

In [3]:
myModel.add(Layer(layer_type="TNN", num_col=1, num_neurons=1, num_dend=1, p_dist=4, p_prox=1, num_seg=4, wres_dist=3, wres_prox=3, thres=6))

In [4]:
myModel.summary()

TNN_Layer_0
     NUM_COL 2
     NUM_NEURONS 2
     NUM_DEND 1
     P_DIST 18
     P_PROX 1
     NUM_SEG 4
     WRES_DIST 3
     WRES_PROX 3
     THRESHOLD 6
TNN_Layer_1
     NUM_COL 1
     NUM_NEURONS 1
     NUM_DEND 1
     P_DIST 4
     P_PROX 1
     NUM_SEG 4
     WRES_DIST 3
     WRES_PROX 3
     THRESHOLD 6


In [5]:
myModel.compile()

In [6]:
myModel.model.get_ports()

OrderedDict([('clk', <veriloggen.core.vtypes.Input at 0x1e874077580>),
             ('grst', <veriloggen.core.vtypes.Input at 0x1e874077640>),
             ('rstb', <veriloggen.core.vtypes.Input at 0x1e874077850>),
             ('model_input', <veriloggen.core.vtypes.Input at 0x1e8741ed070>),
             ('L0_input_spikes_prox_0',
              <veriloggen.core.vtypes.Input at 0x1e8741ed040>),
             ('L0_w_init_dist_0_0_000',
              <veriloggen.core.vtypes.Input at 0x1e8741ed190>),
             ('L0_w_init_dist_0_0_001',
              <veriloggen.core.vtypes.Input at 0x1e8741ed430>),
             ('L0_w_init_dist_0_0_002',
              <veriloggen.core.vtypes.Input at 0x1e8741ed670>),
             ('L0_w_init_dist_0_0_003',
              <veriloggen.core.vtypes.Input at 0x1e8741ed700>),
             ('L0_w_init_dist_0_0_004',
              <veriloggen.core.vtypes.Input at 0x1e8741ed9a0>),
             ('L0_w_init_dist_0_0_005',
              <veriloggen.core.vtypes.Inpu

In [6]:
myModel.add(Layer(layer_type="TNN", num_col=1, num_neurons=1, num_dend=2, p_dist=20, p_prox=1, num_seg=1, wres_dist=3, wres_prox=3, thres=6))

In [2]:
myModel = Model()
myModel.add(Layer(layer_type="TNN", num_col=2, num_neurons=4, num_dend=2, p_dist=2, p_prox=1, num_seg=3, wres_dist=3, wres_prox=3, thres=6))

In [4]:
myModel.add(Layer(layer_type="TNN", num_col=1, num_neurons=4, num_dend=1, p_dist=2, p_prox=1, num_seg=1, wres_dist=3, wres_prox=3, thres=6))

In [5]:
layers = []

In [11]:
myModel.layers

In [13]:
print(myModel.layers[1].name)

TNN_Layer_1


In [4]:
temp = Layers(num_col=2, num_neurons=4, num_dend=2, p_dist=2, p_prox=1, num_seg=3, wres_dist=3, wres_prox=3, thres=6)
layers.append(temp)

In [4]:
temp = Layers.TNN_Layer(num_col=1, num_neurons=4, num_dend=1, p_dist=2, p_prox=1, num_seg=1, wres_dist=3, wres_prox=3, thres=6)
layers.append(temp)

In [5]:
temp = Layers.TNN_Layer(num_col=1, num_neurons=2, num_dend=1, p_dist=2, p_prox=1, num_seg=1, wres_dist=3, wres_prox=3, thres=6)
layers.append(temp)

In [6]:
layers

In [7]:
model = Module('model')
clk = model.Input('clk')
grst = model.Input('grst')
rstb = model.Input('rstb')

In [8]:
# Generate model (wrapper) ports
for i in range(len(layers)):
    layer = layers[i]
    ports = copy.deepcopy(layer.get_ports())
    params = copy.deepcopy(layer.get_params())
    
    # Copy all params to model
    for key in params:
        param = params[key]
        param_name = param.name
        param.name = 'L'+str(i)+'_'+param_name
        model.add_object(param)
        
    # model input ports should match layer 1 input ports
    if (i==0):
        # iterate through ports
        for key in ports:
            port = ports[key]
            port_name = port.name
            # skip clk, grst, rstb
            if ((port_name!='clk') & (port_name!='grst') & (port_name!='rstb')):
                # add input ports to model
                if (isinstance(port, core.vtypes.Input)):
                    if (port_name.startswith('input_spikes_dist')):
                        port.name = 'model_input'
                    else:
                        port.name = 'L0_'+port.name
                    model.add_object(port)
    # add non-connecting input ports to model for other layers
    else:
        # iterate through ports
        for key in ports:
            port = ports[key]
            port_name = port.name
            # skip clk, grst, rstb
            if ((port_name!='clk') & (port_name!='grst') & (port_name!='rstb')):
                # add input ports to model
                if (isinstance(port, core.vtypes.Input)):
                    # filter out connecting ports
                    if (not(port_name.startswith('input_spikes_dist'))):
                        port.name = 'L'+str(i)+'_'+port.name
                        model.add_object(port)
                        
        # generate output port
        if (i == len(layers)-1):
            for key in ports:
                port = ports[key]
                port_name = port.name
                if (isinstance(port, core.vtypes.Output)):
                    port.name = 'model_output'
                    model.add_object(port)

In [9]:
model_ports = model.get_ports()
model_params = model.get_params()

for i in range(len(layers)):
    layer = layers[i]
    layer_ports = [clk, grst, rstb]
    layer_params = [i]
    
    # Add params
    for key in model_params:
        if (key).startswith('L'+str(i)):
            print(key)
            layer_params.append(model_params[key])
    
    # For first layer, add all ports with prefix L0
    if i == 0:
        for key in model_ports:
            if ((key).startswith('L'+str(i)) | (key).startswith('model_input')):
                layer_ports.append(model_ports[key])
                
        # Instantiate wire to connect output to next layer's input
        neuron_count = layers[i].get_params()['NUM_NEURONS'].value * layers[i].get_params()['NUM_COL'].value
        last_out = model.Wire('out_'+str(i)+'_in_'+str(i+1), neuron_count)
        layer_ports.append(last_out)
    else:
        # distal input is last layer's output
        layer_ports.append(last_out)
        
        for key in model_ports:
            if (key).startswith('L'+str(i)):
                layer_ports.append(model_ports[key])
                
        # Instantiate wire to connect output to next layer's input
        if (i != (len(layers)-1)):
            neuron_count = layers[i].get_params()['NUM_NEURONS'].value * layers[i].get_params()['NUM_COL'].value
            last_out = model.Wire('out_'+str(i)+'_in_'+str(i+1), neuron_count)
            layer_ports.append(last_out)
        else:
            layer_ports.append(model_ports['model_output'])

    model.Instance(layer, 'L'+str(i)+'_'+layer.name, params = layer_params,
                  ports = layer_ports)

L0_NUM_COL
L0_NUM_NEURONS
L0_NUM_DEND
L0_P_DIST
L0_P_PROX
L0_NUM_SEG
L0_WRES_DIST
L0_WRES_PROX
L0_THRESHOLD
L1_NUM_COL
L1_NUM_NEURONS
L1_NUM_DEND
L1_P_DIST
L1_P_PROX
L1_NUM_SEG
L1_WRES_DIST
L1_WRES_PROX
L1_THRESHOLD
L2_NUM_COL
L2_NUM_NEURONS
L2_NUM_DEND
L2_P_DIST
L2_P_PROX
L2_NUM_SEG
L2_WRES_DIST
L2_WRES_PROX
L2_THRESHOLD


In [10]:
gen_file, rtl_path = gen_verilog(module = model, filename = 'model.v')

In [41]:
layer = myModel.layers[0]

In [44]:
params = layer.get_params()
param = params['NUM_COL']

In [46]:
param.name

'NUM_COL'

In [2]:
myModel = Model()

In [3]:
myModel.add(Layers.TNN_Layer(num_col=2, num_neurons=4, num_dend=2, p_dist=2, p_prox=1, num_seg=3, wres_dist=3, wres_prox=3, thres=6))

In [4]:
myModel.add(Layers.TNN_Layer(num_col=1, num_neurons=4, num_dend=1, p_dist=2, p_prox=1, num_seg=1, wres_dist=1, wres_prox=1, thres=6))

In [5]:
myModel.add(Layers.TNN_Layer(num_col=1, num_neurons=2, num_dend=1, p_dist=2, p_prox=1, num_seg=1, wres_dist=1, wres_prox=1, thres=6))

In [8]:
myModel.summary()

CV_Layer_0
     NUM_COL 2
     NUM_NEURONS 10
     P_DIST 18
     P_PROX 1
     NUM_SEG 2
     WRES_DIST 3
     WRES_PROX 3
     THRESHOLD 6
TNN_Layer_1
     NUM_COL 1
     NUM_NEURONS 1
     NUM_DEND 2
     P_DIST 20
     P_PROX 1
     NUM_SEG 1
     WRES_DIST 3
     WRES_PROX 3
     THRESHOLD 6


In [9]:
myModel.compile()

In [9]:
myModel.model

In [7]:
gen_file, rtl_path = gen_verilog(module = myModel.model, filename = 'cv_model.v')